# Find author qids using wikidata's GBDE (German Projekt Gutenberg) author IDs

Such IDs are derived from the URL and take the for of a short string derived from the authors last name, see below for some examples. The value is attached to an author entity using the ID property wdt:P7753

In [39]:
import logging
logging.basicConfig(filename='Find_Author_Qid_using_PGDE_id.log', 
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)

logger = logging.getLogger('')
logger.debug('Start logging')

In [1]:
from pathlib import Path
import pandas as pd
from datetime import datetime
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
import json
import re

USERNAME = 'LiteraryWorksMetaDataUploadBot'

wbi_config['USER_AGENT'] = f'{USERNAME} [1.0] (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://www.wikidata.org/w/api.php'

In [41]:
collections_dir = Path("../metadata_collections/")
pg_dir = Path("../data_PG/")

def date_tag():
    return datetime.today().strftime("%Y-%m-%dT%H:%M")

def qid_from_url(url : str):
    return re.search(r'Q\d+$',url).group()

### Load our metadata file

The pgde_author_id contains what wikidata calls GBDE author id. We extracted this value from the URL when we scraped the texts.

In [42]:
#filename = 'de-fiction-wd-url2025-11-13T19:10:33.csv'
filename = 'de_fiction_metadata_2025-11-14T17.csv'
logger.info(f'Opening file: {filename}')
fiction = pd.read_csv(Path(collections_dir, filename), index_col=0, dtype={'gutenberg_id':str})
fiction

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,NaN,1916,https://www.projekt-gutenberg.org/achleitn/fin...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Finanzer.txt,m,1723,24034,PG-DE,NaN,NaN,NaN,NaN,NaN
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,NaN,1903,https://www.projekt-gutenberg.org/achleitn/moo...,achleitn,Q77439,...,Arthur_Achleitner_-_Das_Schloß_im_Moor.txt,m,3460,51072,PG-DE,NaN,NaN,NaN,NaN,NaN
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,NaN,1896,https://www.projekt-gutenberg.org/achleitn/lug...,achleitn,Q77439,...,Arthur_Achleitner_-_Familie_Lugmüller.txt,m,1967,26741,PG-DE,NaN,NaN,NaN,NaN,NaN
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,NaN,1901,https://www.projekt-gutenberg.org/achleitn/bez...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Bezirkshauptmann._Erst...,m,2614,34895,PG-DE,NaN,NaN,NaN,NaN,NaN
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,NaN,1910,https://www.projekt-gutenberg.org/achleitn/ber...,achleitn,Q77439,...,Arthur_Achleitner_-_Geschichten_aus_den_Bergen...,m,6350,127192,PG-DE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,NaN,52478,1912,NaN,NaN,NaN,...,"Zweig,_Arnold_-_Die_Novellen_um_Claudia-52478.txt",m,2720,47335,PG-US,NaN,NaN,NaN,NaN,NaN
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,NaN,57114,1919,NaN,NaN,NaN,...,"Zweig,_Friderike_Maria_Burger_Winternitz_-_Vög...",m,7846,104521,PG-US,NaN,NaN,NaN,NaN,NaN
4231,Stefan Zweig,Zweig,Stefan,Amok: Novellen einer Leidenschaft,Q675609,57850,1922,NaN,zweig,Q78491,...,"Zweig,_Stefan_-_Amok:_Novellen_einer_Leidensch...",m,3087,69716,PG-US,NaN,NaN,NaN,NaN,NaN


### Query Wikidata for all GBDE author IDs

Could do this somewhat more carefully and only ask for one title, or only for German works

In [43]:
query = """SELECT ?author_qid ?author_qidLabel ?gbde_author_id 
        WHERE
        {
        ?author_qid wdt:P7753 ?gbde_author_id .
        SERVICE wikibase:label { bd:serviceParam wikibase:language "de,mul,en". }
        }
        ORDER BY desc(?author_qidLabel) desc(?title)"""

result = wbi_helpers.execute_sparql_query(query, 
                                            user_agent=wbi_config['USER_AGENT'])
result

{'head': {'vars': ['author_qid', 'author_qidLabel', 'gbde_author_id']},
 'results': {'bindings': [{'author_qid': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q692'},
    'gbde_author_id': {'type': 'literal', 'value': 'shakespr'},
    'author_qidLabel': {'xml:lang': 'mul',
     'type': 'literal',
     'value': 'William Shakespeare'}},
   {'author_qid': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q2172949'},
    'gbde_author_id': {'type': 'literal', 'value': 'herzog'},
    'author_qidLabel': {'xml:lang': 'mul',
     'type': 'literal',
     'value': 'Rudolf Herzog'}},
   {'author_qid': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q7245'},
    'gbde_author_id': {'type': 'literal', 'value': 'twain'},
    'author_qidLabel': {'xml:lang': 'mul',
     'type': 'literal',
     'value': 'Mark Twain'}},
   {'author_qid': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q590'},
    'gbde_author_id': {'type': 'literal', 'value': 'camoes

### Make a GBDE author ID to QID dict

In [44]:
author_dict = { stmt['gbde_author_id']['value'] : qid_from_url(stmt['author_qid']['value']) for stmt in result['results']['bindings'] }
author_dict

{'shakespr': 'Q692',
 'herzog': 'Q2172949',
 'twain': 'Q7245',
 'camoes': 'Q590',
 'nerval': 'Q191305',
 'sand': 'Q3816',
 'dante': 'Q1067',
 'darwin': 'Q1035',
 'pasztor': 'Q1242600',
 'lescomba': 'Q20731648',
 'horvath': 'Q84455',
 'boetie': 'Q290227',
 'zola': 'Q504',
 'verhaere': 'Q193680',
 'gaboriau': 'Q463513',
 'erckmann': 'Q1348668',
 'rod': 'Q122356',
 'maynial': 'Q18115592',
 'aesop': 'Q43423',
 'zettl': 'Q191377',
 'nielsen': 'Q12344242',
 'xenophon': 'Q129772',
 'reymont': 'Q121180',
 'garschin': 'Q333222',
 'eschenba': 'Q18821',
 'kirchbac': 'Q87838',
 'hellmert': 'Q23710233',
 'borchert': 'Q58799',
 'mozart': 'Q254',
 'kalckreu': 'Q2588858',
 'baudissi': 'Q215699',
 'baudiswg': 'Q98142',
 'uxkull': 'Q55896154',
 'solitair': 'Q19844040',
 'korolenk': 'Q335064',
 'seidelw': 'Q111110',
 'pastor': 'Q1397844',
 'rudinoff': 'Q84411289',
 'alexis': 'Q77312',
 'thackera': 'Q167768',
 'lequeux': 'Q1232165',
 'lockewj': 'Q3568747',
 'russel': 'Q12407121',
 'yeats': 'Q40213',
 'bol

### Just some book-keeping

In [ ]:
# OLD, already renamed. Also, badly chosen names.

fiction = fiction.rename(axis=1, mapper={'qid': 'author_qid', 'gbde_author_id' : 'pgde_author_id'})
fiction

,author,author_last,author_first,title,wd,gutenberg_id,year,url,pgde_author_id,author_qid
index,,,,,,,,,,
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,NaN,1916,https://www.projekt-gutenberg.org/achleitn/fin...,achleitn,Q77439
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,NaN,1903,https://www.projekt-gutenberg.org/achleitn/moo...,achleitn,Q77439
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,NaN,1896,https://www.projekt-gutenberg.org/achleitn/lug...,achleitn,Q77439
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,NaN,1901,https://www.projekt-gutenberg.org/achleitn/bez...,achleitn,Q77439
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,NaN,1910,https://www.projekt-gutenberg.org/achleitn/ber...,achleitn,Q77439
...,...,...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,NaN,52478,1912,NaN,NaN,NaN
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,NaN,57114,1919,NaN,NaN,NaN
4231,Stefan Zweig,Zweig,Stefan,Amok,Q675609,57850,1922,https://www.projekt-gutenberg.org/zweig/amok/a...,zweig,Q78491


### Now we fill the new column with the newly fetched QIDs

In [46]:
fiction['author_qid'] = fiction['pgde_author_id'].map(author_dict)
fiction

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,NaN,1916,https://www.projekt-gutenberg.org/achleitn/fin...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Finanzer.txt,m,1723,24034,PG-DE,NaN,NaN,NaN,NaN,NaN
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,NaN,1903,https://www.projekt-gutenberg.org/achleitn/moo...,achleitn,Q77439,...,Arthur_Achleitner_-_Das_Schloß_im_Moor.txt,m,3460,51072,PG-DE,NaN,NaN,NaN,NaN,NaN
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,NaN,1896,https://www.projekt-gutenberg.org/achleitn/lug...,achleitn,Q77439,...,Arthur_Achleitner_-_Familie_Lugmüller.txt,m,1967,26741,PG-DE,NaN,NaN,NaN,NaN,NaN
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,NaN,1901,https://www.projekt-gutenberg.org/achleitn/bez...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Bezirkshauptmann._Erst...,m,2614,34895,PG-DE,NaN,NaN,NaN,NaN,NaN
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,NaN,1910,https://www.projekt-gutenberg.org/achleitn/ber...,achleitn,Q77439,...,Arthur_Achleitner_-_Geschichten_aus_den_Bergen...,m,6350,127192,PG-DE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,NaN,52478,1912,NaN,NaN,NaN,...,"Zweig,_Arnold_-_Die_Novellen_um_Claudia-52478.txt",m,2720,47335,PG-US,NaN,NaN,NaN,NaN,NaN
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,NaN,57114,1919,NaN,NaN,NaN,...,"Zweig,_Friderike_Maria_Burger_Winternitz_-_Vög...",m,7846,104521,PG-US,NaN,NaN,NaN,NaN,NaN
4231,Stefan Zweig,Zweig,Stefan,Amok: Novellen einer Leidenschaft,Q675609,57850,1922,NaN,zweig,Q78491,...,"Zweig,_Stefan_-_Amok:_Novellen_einer_Leidensch...",m,3087,69716,PG-US,NaN,NaN,NaN,NaN,NaN


In [ ]:
fiction.to_csv(Path(collections_dir, 'de-fiction-pgde-auth-id.csv'))

#filename = 'de_fiction_metadata_2025-11-14T18.csv'
filename = f'de_fiction_metadata-{date_tag()}.csv'
logger.info(f'Writing file: {filename}')
fiction.to_csv(Path(collections_dir, filename))